<a href="https://colab.research.google.com/github/maheshkrrs008-sudo/Agentic-AI-Training/blob/main/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

# ⚠️ Important: If running in Colab , Switch to Python 3.12

Before running any other cells in this notebook, please switch your Google Colab runtime to **Python 3.12**.

From the top right arrow (next to Connect/RAM), select:

**Change runtime type → Runtime Version → 2026.07**

The next cell should output python version as 3.12


In [2]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   TAVILY_API_KEY

> 💡 **Note:** While running package installation command below,Google Colab may display some dependency messages during installation. These relate to packages we are not using in this notebook, so you can safely continue when the installation completes.

In [3]:
!pip install -q \
    "langchain==0.3.14" \
    "langchain-openai==0.2.14" \
    "langchain-community==0.3.14" \
    "langchain-core==0.3.63" \
    "openai==1.59.6" \
    "python-dotenv==1.0.1" \
    "requests==2.32.4" \
    "beautifulsoup4>=4.13.0" \
    "wikipedia==1.4.0" \
    "tavily-python==0.5.0"

In [ ]:
#below lines are needed for restarting kernel in Colab -
# If you are in local environment you can use "restart" button on top of the notebook
import os
os.kill(os.getpid(), 9)

In [1]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [2]:
import requests
from bs4 import BeautifulSoup

In [3]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

Hello, David! How can I help you today?


In [6]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [7]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple custom tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create tool from function
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)

# Tavily Search Tool
tavily_search = TavilySearchResults(max_results=2)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Tools list
tools = [tavily_search, my_tool]

# Create agent
agent = initialize_agent(
    tools,
    llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("What's the weather like today in London?")

print(response)

In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [tavily_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [ ]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])